In [1]:
import os
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = '/content/drive/MyDrive/sales.csv'
    OUT_DIR = '/content/drive/MyDrive'
except ImportError:
    DATA_PATH = 'sales.csv'
    OUT_DIR = 'results'
os.makedirs(OUT_DIR, exist_ok=True)

Mounted at /content/drive


In [2]:
def make_demo_sales(n=3636, seed=42):
    rng = np.random.default_rng(seed)
    prices = {'Latte': 38.7, 'Hot Chocolate': 38.7, 'Americano': 28.9,
              'Americano with Milk': 33.8, 'Cappuccino': 38.7, 'Cocoa': 35.76,
              'Cortado': 30.0, 'Espresso': 25.96}
    names = rng.choice(list(prices), size=n)
    start, end = pd.Timestamp('2024-03-01 10:00'), pd.Timestamp('2025-03-23 19:00')
    dt = pd.Series(pd.to_datetime(np.sort(rng.integers(start.value, end.value, size=n))))
    cash_type = np.where(rng.random(n) < 0.9, 'card', 'cash')
    card = [f'ANON-0000-0000-{i:04d}' if t == 'card' else None
            for t, i in zip(cash_type, rng.integers(1, 1200, size=n))]
    return pd.DataFrame({
        'date': dt.dt.strftime('%Y-%m-%d'),
        'datetime': dt.dt.strftime('%Y-%m-%d %H:%M:%S.%f').str[:-3],
        'cash_type': cash_type,
        'card': card,
        'money': [prices[x] for x in names],
        'coffee_name': names,
    })

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
else:
    print(f'Файл {DATA_PATH} не знайдено - використовую демонстраційні дані.')
    df = make_demo_sales()

df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
df['day_of_week'] = df['datetime'].dt.dayofweek
df['hour'] = df['datetime'].dt.hour
df.head()

,date,datetime,cash_type,card,money,coffee_name,day_of_week,hour
0,2024-03-01,2024-03-01 11:58:48.618,card,ANON-0000-0000-0450,33.80,Hot chocolate,4,11
1,2024-03-01,2024-03-01 13:43:22.828,card,ANON-0000-0000-0207,28.90,Americano,4,13
2,2024-03-01,2024-03-01 14:33:30.639,cash,NaN,35.76,Cocoa,4,14
3,2024-03-01,2024-03-01 14:52:46.593,card,ANON-0000-0000-0484,33.80,Americano with milk,4,14
4,2024-03-01,2024-03-01 15:16:40.466,card,ANON-0000-0000-0285,38.70,Latte,4,15


In [3]:
trans_by_cash_type = df.groupby('cash_type').size()
print(trans_by_cash_type)
print(type(trans_by_cash_type))

cash_type
card    903
cash     97
dtype: int64
<class 'pandas.core.series.Series'>


In [4]:
min_max = df.groupby('coffee_name')['money'].agg(['max', 'min'])
print(min_max)

                       max    min
coffee_name                      
Americano            28.90  28.90
Americano with milk  33.80  33.80
Cappuccino           38.70  38.70
Cocoa                35.76  35.76
Cortado              29.00  29.00
Espresso             24.00  24.00
Hot chocolate        33.80  33.80
Latte                38.70  38.70


In [5]:
pivot_task3 = pd.pivot_table(df, values='money', index='cash_type',
                             columns='coffee_name', aggfunc='sum', fill_value=0)
pivot_task3

coffee_name,Americano,Americano with milk,Cappuccino,Cocoa,Cortado,Espresso,Hot chocolate,Latte
cash_type,,,,,,,,
card,5288.7,3413.8,5185.8,1895.28,1189.0,1464.0,2805.4,9558.9
cash,809.2,371.8,464.4,321.84,261.0,168.0,169.0,619.2


In [6]:
crosstab_task4 = pd.crosstab(df['day_of_week'], df['cash_type'])
crosstab_task4

cash_type,card,cash
day_of_week,,
0,128,12
1,119,14
2,131,11
3,121,16
4,130,12
5,154,17
6,120,15


Рядки — дні тижня (0 = понеділок, 6 = неділя), стовпці — типи оплати, значення — кількість транзакцій.

In [7]:
costs = pd.DataFrame({
    'coffee_name': ['Latte', 'Americano', 'Americano with Milk', 'Cappuccino',
                    'Cocoa', 'Hot Chocolate', 'Cortado', 'Espresso'],
    'cost': [15, 10, 13, 14, 12, 13, 12, 9]
})

df_cost = df.merge(costs, on='coffee_name', how='left')
df_cost['profit'] = df_cost['money'] - df_cost['cost']

print(df_cost[['coffee_name', 'money', 'cost', 'profit']].head(10))
print(df_cost.groupby('coffee_name')['profit'].sum().round(2))

           coffee_name  money  cost  profit
0        Hot chocolate  33.80   NaN     NaN
1            Americano  28.90  10.0   18.90
2                Cocoa  35.76  12.0   23.76
3  Americano with milk  33.80   NaN     NaN
4                Latte  38.70  15.0   23.70
5              Cortado  29.00  12.0   17.00
6        Hot chocolate  33.80   NaN     NaN
7        Hot chocolate  33.80   NaN     NaN
8                Latte  38.70  15.0   23.70
9            Americano  28.90  10.0   18.90
coffee_name
Americano              3987.90
Americano with milk       0.00
Cappuccino             3606.20
Cocoa                  1473.12
Cortado                 850.00
Espresso               1020.00
Hot chocolate             0.00
Latte                  6233.10
Name: profit, dtype: float64


In [8]:
part_a = pd.DataFrame({'product': ['tea', 'coffee', 'cocoa'], 'qty': [3, 5, 2]}, index=[0, 1, 2])
part_b = pd.DataFrame({'product': ['juice', 'water', 'milk'], 'qty': [7, 1, 4]}, index=[3, 4, 5])

combined = pd.concat([part_a, part_b])
print(combined)
print('Індекси збережено:', list(combined.index) == list(part_a.index) + list(part_b.index))

  product  qty
0     tea    3
1  coffee    5
2   cocoa    2
3   juice    7
4   water    1
5    milk    4
Індекси збережено: True


In [9]:
part_c = pd.DataFrame({'product': ['bread', 'butter', 'jam'], 'qty': [9, 2, 6]}, index=[0, 1, 2])

combined_same = pd.concat([part_a, part_c])
print(combined_same)
print(combined_same.index.tolist())
print(pd.concat([part_a, part_c], ignore_index=True).index.tolist())

  product  qty
0     tea    3
1  coffee    5
2   cocoa    2
0   bread    9
1  butter    2
2     jam    6
[0, 1, 2, 0, 1, 2]
[0, 1, 2, 3, 4, 5]


`concat` за замовчуванням зберігає початкові індекси, навіть якщо вони повторюються; щоб отримати нову суцільну нумерацію, використовують `ignore_index=True`.

In [10]:
avg_check_by_hour = df.groupby('hour')['money'].mean().round(2)
print(avg_check_by_hour)

hour
7     34.03
8     33.84
9     33.75
10    34.02
11    35.42
12    34.41
13    33.75
14    34.44
15    33.76
16    34.28
17    34.33
18    33.32
19    33.00
20    33.61
21    32.19
Name: money, dtype: float64


In [11]:
rng = np.random.default_rng(11)

products = pd.DataFrame({
    'product_id': [1, 2, 3, 4, 5, 6],
    'product': ['Tea', 'Coffee', 'Cocoa', 'Croissant', 'Sandwich', 'Cake'],
    'category': ['Drink', 'Drink', 'Drink', 'Food', 'Food', 'Food'],
    'unit_price': [25.0, 35.0, 40.0, 30.0, 55.0, 60.0]
})

stores = pd.DataFrame({
    'store_id': [1, 2, 3],
    'city': ['Kyiv', 'Lviv', 'Odesa']
})

orders_jan = pd.DataFrame({
    'order_id': range(1, 101),
    'month': 'January',
    'store_id': rng.integers(1, 4, 100),
    'product_id': rng.integers(1, 7, 100),
    'quantity': rng.integers(1, 6, 100)
})

orders_feb = pd.DataFrame({
    'order_id': range(101, 181),
    'month': 'February',
    'store_id': rng.integers(1, 4, 80),
    'product_id': rng.integers(1, 7, 80),
    'quantity': rng.integers(1, 6, 80)
})

orders = pd.concat([orders_jan, orders_feb], ignore_index=True)
print(orders.shape)

(180, 5)


In [12]:
data = orders.merge(products, on='product_id', how='left').merge(stores, on='store_id', how='left')
data['revenue'] = data['quantity'] * data['unit_price']
data.head()

,order_id,month,store_id,product_id,quantity,product,category,unit_price,city,revenue
0,1,January,1,1,1,Tea,Drink,25.0,Kyiv,25.0
1,2,January,1,3,1,Cocoa,Drink,40.0,Kyiv,40.0
2,3,January,3,5,4,Sandwich,Food,55.0,Odesa,220.0
3,4,January,2,5,3,Sandwich,Food,55.0,Lviv,165.0
4,5,January,2,6,3,Cake,Food,60.0,Lviv,180.0


In [13]:
summary_bonus = data.groupby(['city', 'category']).agg(
    total_revenue=('revenue', 'sum'),
    avg_revenue=('revenue', 'mean'),
    orders_count=('order_id', 'count')
).round(2).reset_index()
summary_bonus

,city,category,total_revenue,avg_revenue,orders_count
0,Kyiv,Drink,2780.0,86.88,32
1,Kyiv,Food,4050.0,155.77,26
2,Lviv,Drink,2820.0,85.45,33
3,Lviv,Food,4035.0,155.19,26
4,Odesa,Drink,3200.0,106.67,30
5,Odesa,Food,4600.0,139.39,33


In [14]:
pivot_bonus = pd.pivot_table(data, values='revenue', index='city',
                             columns='category', aggfunc='sum', fill_value=0)
pivot_bonus

category,Drink,Food
city,,
Kyiv,2780.0,4050.0
Lviv,2820.0,4035.0
Odesa,3200.0,4600.0


In [15]:
bonus_dir = os.path.join(OUT_DIR, 'bonus')
os.makedirs(bonus_dir, exist_ok=True)

data.to_csv(os.path.join(bonus_dir, 'bonus_merged.csv'), index=False)
summary_bonus.to_csv(os.path.join(bonus_dir, 'bonus_summary.csv'), index=False)
pivot_bonus.to_csv(os.path.join(bonus_dir, 'bonus_pivot.csv'))
print(sorted(os.listdir(bonus_dir)))

['bonus_merged.csv', 'bonus_pivot.csv', 'bonus_summary.csv']
